# 11. 模型 Calibration 与 Threshold：怎样用 Temperature、ECE 和业务成本共同定决策？

## 面试回答主线

分类 logit 的排序可以正确，但 sigmoid 概率仍可能过度自信；校准要让预测 0.8 的样本长期约有 80% 为正例。Temperature Scaling 学习一个正数 T，用 `logit / T` 调整置信度而不改变排序。面试时我会在真实风控工单上手写 Brier、ECE 分桶，使用 PyTorch 梯度优化 `log_temperature`，再输出训练前后逐样本概率。阈值不是固定 0.5，应根据漏放与误报成本在校准概率上选择，并保留逐阈值成本表。全局 temperature 还可能掩盖白天/夜间分布差异，需要 slice 监控或在样本充足时分组校准。生产校准必须在独立 validation 上拟合，并在未来 holdout 上验收。

## 1. 真实案例：十八笔风控工单的 logit、标签与时段

白天十二笔、夜间六笔；原模型对多数样本方向正确，但在少数误判上给出极端 logit，构成典型过度自信。标签 1 表示需要人工审核。

In [1]:
from pprint import pprint  # 导入结构化打印工具展示概率、分桶和阈值结果
import torch  # 导入 PyTorch 执行真实 temperature 梯度优化
from torch.nn import functional as F  # 导入稳定 BCE-with-logits 校准损失
torch.set_num_threads(1)  # 限制教学训练线程数以保持执行稳定
torch.manual_seed(19)  # 固定 temperature 参数优化轨迹
raw_logits = torch.tensor([3.0, 2.0, 1.5, 1.0, 0.5, -0.5, -1.0, -1.5, -2.0, -3.0, 2.5, -2.5, 4.0, 3.0, 2.0, -2.0, -3.0, -4.0], dtype=torch.float32)  # 定义十八笔工单的未校准模型 logits
labels = torch.tensor([1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0], dtype=torch.float32)  # 定义是否需要人工审核的真实标签
segments = ["day"] * 12 + ["night"] * 6  # 标记白天与夜间业务分布切片
case_ids = [f"C{index:02d}" for index in range(1, len(labels) + 1)]  # 为十八笔工单生成稳定样本 ID
preview = [{"工单": case_id, "时段": segments[index], "raw_logit": float(raw_logits[index]), "标签": int(labels[index])} for index, case_id in enumerate(case_ids)]  # 汇总校准所需业务字段
print("Calibration 工单预览：")  # 输出真实案例标题
pprint(preview, sort_dicts=False)  # 展示正负样本、极端 logit 和时段分布

Calibration 工单预览：
[{'工单': 'C01', '时段': 'day', 'raw_logit': 3.0, '标签': 1},
 {'工单': 'C02', '时段': 'day', 'raw_logit': 2.0, '标签': 1},
 {'工单': 'C03', '时段': 'day', 'raw_logit': 1.5, '标签': 1},
 {'工单': 'C04', '时段': 'day', 'raw_logit': 1.0, '标签': 1},
 {'工单': 'C05', '时段': 'day', 'raw_logit': 0.5, '标签': 0},
 {'工单': 'C06', '时段': 'day', 'raw_logit': -0.5, '标签': 0},
 {'工单': 'C07', '时段': 'day', 'raw_logit': -1.0, '标签': 0},
 {'工单': 'C08', '时段': 'day', 'raw_logit': -1.5, '标签': 0},
 {'工单': 'C09', '时段': 'day', 'raw_logit': -2.0, '标签': 0},
 {'工单': 'C10', '时段': 'day', 'raw_logit': -3.0, '标签': 0},
 {'工单': 'C11', '时段': 'day', 'raw_logit': 2.5, '标签': 0},
 {'工单': 'C12', '时段': 'day', 'raw_logit': -2.5, '标签': 1},
 {'工单': 'C13', '时段': 'night', 'raw_logit': 4.0, '标签': 1},
 {'工单': 'C14', '时段': 'night', 'raw_logit': 3.0, '标签': 0},
 {'工单': 'C15', '时段': 'night', 'raw_logit': 2.0, '标签': 1},
 {'工单': 'C16', '时段': 'night', 'raw_logit': -2.0, '标签': 0},
 {'工单': 'C17', '时段': 'night', 'raw_logit': -3.0, '标签': 1},
 {'工单': 'C18

## 2. Baseline（基线）：原始 sigmoid 的 Brier 与 ECE 分桶

Brier 是概率与标签平方误差均值；ECE 把概率分五桶，对每桶置信度与正例率差取样本加权平均。下面手写两个指标并保存每桶样本数、平均概率和实际正例率。

In [2]:
def calibration_metrics(probabilities, targets, bin_count=5):  # 手写 Brier、ECE 和可靠性分桶统计
    brier = float(torch.mean((probabilities - targets) ** 2))  # 计算逐样本概率平方误差均值
    rows = []  # 收集每个概率桶的置信度与真实频率
    ece = 0.0  # 初始化 expected calibration error
    for bin_index in range(bin_count):  # 遍历从零到一的等宽概率桶
        lower = bin_index / bin_count  # 计算当前桶左边界
        upper = (bin_index + 1) / bin_count  # 计算当前桶右边界
        mask = (probabilities >= lower) & (probabilities < upper if bin_index < bin_count - 1 else probabilities <= upper)  # 选择落入当前桶的样本
        count = int(mask.sum())  # 统计当前桶样本数
        if count > 0:  # 只有非空桶才能计算经验概率
            confidence = float(probabilities[mask].mean())  # 计算模型在当前桶的平均预测概率
            positive_rate = float(targets[mask].mean())  # 计算当前桶真实正例比例
            gap = abs(confidence - positive_rate)  # 计算置信度与实际频率差
            ece += count / len(targets) * gap  # 按样本占比累加 ECE
            rows.append({"区间": f"[{lower:.1f},{upper:.1f}]", "样本数": count, "平均概率": confidence, "正例率": positive_rate, "gap": gap})  # 保存可靠性图所需中间量
    return brier, ece, rows  # 返回两个标量指标和非空桶账本
baseline_probabilities = torch.sigmoid(raw_logits)  # 将原始 logits 转换为未校准概率
baseline_brier, baseline_ece, baseline_bins = calibration_metrics(baseline_probabilities, labels)  # 计算原模型概率质量
print("未校准概率的可靠性分桶：")  # 标注当前输出属于基线
pprint([{**row, "平均概率": round(row["平均概率"], 3), "正例率": round(row["正例率"], 3), "gap": round(row["gap"], 3)} for row in baseline_bins], sort_dicts=False)  # 展示每桶概率和真实频率差
print({"Baseline Brier": round(baseline_brier, 4), "Baseline ECE": round(baseline_ece, 4)})  # 汇总未校准概率指标

未校准概率的可靠性分桶：
[{'区间': '[0.0,0.2]', '样本数': 7, '平均概率': 0.087, '正例率': 0.286, 'gap': 0.199},
 {'区间': '[0.2,0.4]', '样本数': 2, '平均概率': 0.323, '正例率': 0.0, 'gap': 0.323},
 {'区间': '[0.6,0.8]', '样本数': 2, '平均概率': 0.677, '正例率': 0.5, 'gap': 0.177},
 {'区间': '[0.8,1.0]', '样本数': 7, '平均概率': 0.913, '正例率': 0.714, 'gap': 0.199}]
{'Baseline Brier': 0.2403, 'Baseline ECE': 0.2101}


## 3. PyTorch Temperature Scaling：优化正数 T 的真实梯度

用 `T = exp(log_temperature)` 保证温度始终为正。目标是最小化 calibration 集上的 binary cross entropy；T 大于 1 会收缩过度极端的 logits，但保持样本排序不变。

In [3]:
log_temperature = torch.nn.Parameter(torch.tensor(0.0))  # 用可训练对数参数表示始终为正的 temperature
optimizer = torch.optim.Adam([log_temperature], lr=0.05)  # 创建只更新一个校准参数的优化器
loss_history = []  # 记录校准 BCE 的优化轨迹
first_gradient = 0.0  # 初始化首步 temperature 梯度
for step in range(400):  # 对十八笔 calibration 样本执行真实梯度优化
    temperature = torch.exp(log_temperature)  # 将无约束参数转换为正数温度
    calibrated_logits = raw_logits / temperature  # 用同一温度缩放所有模型 logits
    loss = F.binary_cross_entropy_with_logits(calibrated_logits, labels)  # 计算校准概率的负对数似然
    optimizer.zero_grad()  # 清除上一轮 temperature 梯度
    loss.backward()  # 通过 logit 除法执行真实反向传播
    if step == 0:  # 在第一次更新前记录非零梯度证据
        first_gradient = float(log_temperature.grad)  # 读取过度自信数据推动温度变化的梯度
    optimizer.step()  # 根据 calibration loss 更新 log temperature
    loss_history.append(float(loss.detach()))  # 保存当前损失供收敛比较
learned_temperature = float(torch.exp(log_temperature).detach())  # 读取训练后的正数温度
calibrated_probabilities = torch.sigmoid(raw_logits / learned_temperature)  # 生成缩放后的逐样本概率
print({"首步log_T梯度": round(first_gradient, 4), "初始NLL": round(loss_history[0], 4), "最终NLL": round(loss_history[-1], 4), "学习到的T": round(learned_temperature, 4), "排序保持": torch.equal(torch.argsort(raw_logits), torch.argsort(raw_logits / learned_temperature))})  # 展示真实梯度、收敛和排序不变量

{'首步log_T梯度': -0.444, '初始NLL': 0.7985, '最终NLL': 0.6216, '学习到的T': 2.9776, '排序保持': True}


## 4. 校准后逐样本概率、ECE 与 Brier

温度不改变 logit 正负和排名，却把 0.98、0.02 等极端概率拉回更符合错误率的范围。逐样本表保留 raw 与 calibrated 概率，分桶表解释 ECE 改进来源。

In [4]:
calibrated_brier, calibrated_ece, calibrated_bins = calibration_metrics(calibrated_probabilities, labels)  # 计算温度缩放后的概率质量
probability_rows = [{"工单": case_ids[index], "时段": segments[index], "标签": int(labels[index]), "raw概率": round(float(baseline_probabilities[index]), 4), "校准概率": round(float(calibrated_probabilities[index]), 4)} for index in range(len(labels))]  # 构造逐样本概率变化表
print("Temperature Scaling 逐工单结果：")  # 输出核心方案结果标题
pprint(probability_rows, sort_dicts=False)  # 展示十八笔工单的置信度收缩
print("校准后可靠性分桶：")  # 输出关键中间量标题
pprint([{**row, "平均概率": round(row["平均概率"], 3), "正例率": round(row["正例率"], 3), "gap": round(row["gap"], 3)} for row in calibrated_bins], sort_dicts=False)  # 展示校准后每桶概率与真实频率
print({"Brier": (round(baseline_brier, 4), round(calibrated_brier, 4)), "ECE": (round(baseline_ece, 4), round(calibrated_ece, 4))})  # 对照校准前后两个指标

Temperature Scaling 逐工单结果：
[{'工单': 'C01', '时段': 'day', '标签': 1, 'raw概率': 0.9526, '校准概率': 0.7325},
 {'工单': 'C02', '时段': 'day', '标签': 1, 'raw概率': 0.8808, '校准概率': 0.6619},
 {'工单': 'C03', '时段': 'day', '标签': 1, 'raw概率': 0.8176, '校准概率': 0.6233},
 {'工单': 'C04', '时段': 'day', '标签': 1, 'raw概率': 0.7311, '校准概率': 0.5832},
 {'工单': 'C05', '时段': 'day', '标签': 0, 'raw概率': 0.6225, '校准概率': 0.5419},
 {'工单': 'C06', '时段': 'day', '标签': 0, 'raw概率': 0.3775, '校准概率': 0.4581},
 {'工单': 'C07', '时段': 'day', '标签': 0, 'raw概率': 0.2689, '校准概率': 0.4168},
 {'工单': 'C08', '时段': 'day', '标签': 0, 'raw概率': 0.1824, '校准概率': 0.3767},
 {'工单': 'C09', '时段': 'day', '标签': 0, 'raw概率': 0.1192, '校准概率': 0.3381},
 {'工单': 'C10', '时段': 'day', '标签': 0, 'raw概率': 0.0474, '校准概率': 0.2675},
 {'工单': 'C11', '时段': 'day', '标签': 0, 'raw概率': 0.9241, '校准概率': 0.6984},
 {'工单': 'C12', '时段': 'day', '标签': 1, 'raw概率': 0.0759, '校准概率': 0.3016},
 {'工单': 'C13', '时段': 'night', '标签': 1, 'raw概率': 0.982, '校准概率': 0.793},
 {'工单': 'C14', '时段': 'night', '标签': 0, 'raw概率': 0.

## 5. 结果解读：业务阈值由漏放成本 5、误报成本 1 决定

在校准概率上枚举 0.1～0.9。每个 false negative 成本为 5，false positive 成本为 1；最优阈值来自完整成本表，而不是默认 0.5。逐样本结果展示最终审核/放行决策。

In [5]:
threshold_rows = []  # 收集候选概率阈值的业务成本
for threshold_index in range(1, 10):  # 枚举九个易解释的阈值候选
    threshold = threshold_index / 10  # 将整数转换为零点一到零点九概率门槛
    decisions = (calibrated_probabilities >= threshold).float()  # 根据校准概率生成是否人工审核决策
    false_negatives = int(((decisions == 0) & (labels == 1)).sum())  # 统计高成本漏放正例
    false_positives = int(((decisions == 1) & (labels == 0)).sum())  # 统计低成本误报审核
    cost = false_negatives * 5 + false_positives * 1  # 按业务损失权重计算总成本
    threshold_rows.append({"threshold": threshold, "FN": false_negatives, "FP": false_positives, "总成本": cost})  # 保存每个阈值的完整混淆成本
best_threshold_row = min(threshold_rows, key=lambda row: (row["总成本"], row["threshold"]))  # 选择成本最低且并列时更保守的较低阈值
best_threshold = best_threshold_row["threshold"]  # 读取最终部署候选阈值
final_decisions = calibrated_probabilities >= best_threshold  # 用选择后的阈值生成逐工单决策
decision_rows = [{"工单": case_ids[index], "校准概率": round(float(calibrated_probabilities[index]), 4), "标签": int(labels[index]), "决策": "人工审核" if final_decisions[index] else "自动放行", "正确": int(final_decisions[index]) == int(labels[index])} for index in range(len(labels))]  # 保存逐样本业务结果
print("阈值与业务成本：")  # 输出阈值搜索中间量标题
pprint(threshold_rows, sort_dicts=False)  # 展示漏放和误报如何共同决定门槛
print("选择阈值后的逐工单决策：")  # 输出结果解读标题
pprint(decision_rows, sort_dicts=False)  # 展示最终概率如何转成业务动作

阈值与业务成本：
[{'threshold': 0.1, 'FN': 0, 'FP': 10, '总成本': 10},
 {'threshold': 0.2, 'FN': 0, 'FP': 10, '总成本': 10},
 {'threshold': 0.3, 'FN': 1, 'FP': 8, '总成本': 13},
 {'threshold': 0.4, 'FN': 2, 'FP': 5, '总成本': 15},
 {'threshold': 0.5, 'FN': 2, 'FP': 3, '总成本': 13},
 {'threshold': 0.6, 'FN': 3, 'FP': 2, '总成本': 17},
 {'threshold': 0.7, 'FN': 6, 'FP': 1, '总成本': 31},
 {'threshold': 0.8, 'FN': 8, 'FP': 0, '总成本': 40},
 {'threshold': 0.9, 'FN': 8, 'FP': 0, '总成本': 40}]
选择阈值后的逐工单决策：
[{'工单': 'C01', '校准概率': 0.7325, '标签': 1, '决策': '人工审核', '正确': True},
 {'工单': 'C02', '校准概率': 0.6619, '标签': 1, '决策': '人工审核', '正确': True},
 {'工单': 'C03', '校准概率': 0.6233, '标签': 1, '决策': '人工审核', '正确': True},
 {'工单': 'C04', '校准概率': 0.5832, '标签': 1, '决策': '人工审核', '正确': True},
 {'工单': 'C05', '校准概率': 0.5419, '标签': 0, '决策': '人工审核', '正确': False},
 {'工单': 'C06', '校准概率': 0.4581, '标签': 0, '决策': '人工审核', '正确': False},
 {'工单': 'C07', '校准概率': 0.4168, '标签': 0, '决策': '人工审核', '正确': False},
 {'工单': 'C08', '校准概率': 0.3767, '标签': 0, '决策': '人工审核', 

## 6. 失败案例与修正：全局 Temperature 掩盖夜间分布差异

夜间日志更极端且错误更多，最优温度比白天大。只报告全局 Brier 会看不到 slice 的剩余过度自信；下面分别拟合 day/night 温度并比较加权分组 Brier。生产中只有样本量足够且独立验证有效时才采用分组模型，否则应至少监控。

In [6]:
def fit_temperature(indices):  # 为一个业务切片拟合独立正数 temperature
    slice_log_temperature = torch.nn.Parameter(torch.tensor(0.0))  # 初始化当前切片的对数温度
    slice_optimizer = torch.optim.Adam([slice_log_temperature], lr=0.05)  # 创建只更新切片温度的优化器
    for step in range(400):  # 在当前切片样本上执行真实梯度校准
        slice_temperature = torch.exp(slice_log_temperature)  # 将切片参数转换为正数温度
        slice_loss = F.binary_cross_entropy_with_logits(raw_logits[indices] / slice_temperature, labels[indices])  # 计算切片概率 NLL
        slice_optimizer.zero_grad()  # 清除上一轮切片温度梯度
        slice_loss.backward()  # 反向传播当前切片的校准误差
        slice_optimizer.step()  # 更新当前切片的 temperature
    return float(torch.exp(slice_log_temperature).detach())  # 返回训练后的正数切片温度
day_indices = torch.tensor([index for index, segment in enumerate(segments) if segment == "day"], dtype=torch.long)  # 收集白天工单索引
night_indices = torch.tensor([index for index, segment in enumerate(segments) if segment == "night"], dtype=torch.long)  # 收集夜间工单索引
day_temperature = fit_temperature(day_indices)  # 拟合白天流量的置信度尺度
night_temperature = fit_temperature(night_indices)  # 拟合夜间流量的置信度尺度
global_night_brier = float(torch.mean((calibrated_probabilities[night_indices] - labels[night_indices]) ** 2))  # 计算全局温度在夜间切片的 Brier
group_probabilities = calibrated_probabilities.clone()  # 初始化将被分组温度覆盖的概率向量
group_probabilities[day_indices] = torch.sigmoid(raw_logits[day_indices] / day_temperature)  # 应用白天专属温度
group_probabilities[night_indices] = torch.sigmoid(raw_logits[night_indices] / night_temperature)  # 应用夜间专属温度
group_brier = float(torch.mean((group_probabilities - labels) ** 2))  # 计算分组温度的整体 Brier
print({"全局T": round(learned_temperature, 3), "白天T": round(day_temperature, 3), "夜间T": round(night_temperature, 3), "全局温度夜间Brier": round(global_night_brier, 4), "全局整体Brier": round(calibrated_brier, 4), "分组温度整体Brier": round(group_brier, 4), "修正": "按时段监控，样本充足后分组校准"})  # 展示 slice 漂移与分组修正

{'全局T': 2.978, '白天T': 2.031, '夜间T': 4.617, '全局温度夜间Brier': 0.2313, '全局整体Brier': 0.215, '分组温度整体Brier': 0.2093, '修正': '按时段监控，样本充足后分组校准'}


## 7. 生产差距与最小回归检查

生产必须把 base model 固定，在独立 validation 上拟合 calibration，再用未来 holdout 评估 ECE/Brier 和业务成本。ECE 对分桶敏感，还应看 reliability curve、adaptive ECE 与 slice；阈值要结合容量、延迟和人工审核预算。下面的断言只验证本实验的真实梯度、正温度、指标改善、阈值和时段差异。

In [7]:
assert len(labels) >= 6  # 确认真实校准样本数量满足逐样本教学要求
assert abs(first_gradient) > 0.0 and learned_temperature > 0.0  # 确认 temperature 通过真实梯度学习且始终为正
assert loss_history[-1] < loss_history[0]  # 确认校准 NLL 在优化过程中下降
assert calibrated_brier < baseline_brier and calibrated_ece < baseline_ece  # 确认 Brier 与 ECE 同时改善
assert best_threshold != 0.5 and best_threshold_row["总成本"] == min(row["总成本"] for row in threshold_rows)  # 确认业务门槛来自成本搜索而非默认值
assert night_temperature > day_temperature  # 确认夜间过度自信需要更强概率收缩
assert group_brier < calibrated_brier  # 确认分组温度在受控时段差异上进一步降低 Brier
print("回归检查通过：Temperature 梯度、ECE/Brier、成本阈值与时段切片均已验证。")  # 输出最终验收结论

回归检查通过：Temperature 梯度、ECE/Brier、成本阈值与时段切片均已验证。
